# 🏄 Surf Spot Search Taiwan: prototype is running

Hi all! Quick update on our IR project. **The prototype works end to end on real data.**

---

## 🔎 What the tool does

You type an English question like:

> *"beginner-friendly beach break near Kaohsiung this weekend"*

The tool returns a **ranked list of Taiwan surf spots**. For each spot it says whether and when it is surfable, e.g.:

> *"Fair for beginners. Best around Sat 06:00: 0.5 m swell, light wind, low tide."*

## 🌊 Where the information comes from

### Live conditions (fetched daily and saved as a *snapshot*)

| Source | What it provides | Notes |
|---|---|---|
| **Open-Meteo Marine API** | 7-day hourly forecast of wave height, swell and wind-wave height/period/direction, and sea level including tides | Free, no key; attribution to Open-Meteo and DWD is required |
| **Open-Meteo Weather API** | Wind speed and direction at each beach | Used for offshore vs. onshore wind |
| **CWA Open Data** (Central Weather Administration) | Real buoy and tide-station measurements from the last 48 h | Used to check the forecast and to measure forecast error. Needs a free API key. English station names come from CWA's official station list |

### Static data (our corpus and rules)

| Data | Sources |
|---|---|
| **Spot profiles** (35 spots) | Mainly **Swelleye** (Taiwan surf-forecast site), plus **BreakFinder**, **NSYSU** (for NSYSU Beach), travel/surf guides (Taiwan Outdoors, WSL, Nick Kembel…) and **OpenStreetMap** for county/town. Every spot lists its sources in `spots.csv` |
| **Surfability thresholds** (e.g. *max wave for beginners = 1.0 m*) | Surf-education sites (Climeradar, Rapture Camps, Sharp Eye…), the **Met Office Beaufort scale** for wind, **Open-Meteo** for sunrise/sunset. Rows without a source are marked **DESIGN CHOICE** with a reason |
| **Surf slang** (*newbie → beginner*, *beachie → beach break*) | **Wiktionary** and the **Wikipedia Glossary of surfing** |
| **Place names and spellings** (*Kaohsiung = Gaoxiong = Takao*) | **Wikidata** |
| **Regions** (*"east coast"*, *"down south"*) | **Wikipedia "Regions of Taiwan"** (official NDC definition) |

> 📚 All source links are inside the CSV files, so we can cite them in the report.

## 📊 Current data

- **35 real spots** across Taiwan (e.g. Wai'ao, Jinzun, Jialeshuei, Cijin, NSYSU Beach)
- **First daily data snapshot:** 16 Oct

## ⚖️ Baseline vs. Improved

| | **Baseline (BM25)** | **Improved (condition-aware)** |
|---|---|---|
| How it ranks | Classic keyword search over the spot descriptions | BM25 **plus** the four additions below |
| Understands place, skill level and time in the query | ❌ | ✅ |
| Translates surf slang | ❌ | ✅ |
| Scores each spot against the forecast for your level and time window | ❌ | ✅ |
| Pushes spots that are unsafe for your level to the bottom | ❌ | ✅ |
| Explains every result | ❌ | ✅ |

The baseline only matches words; it doesn't know where you are, when you want to surf, or what the waves are doing.

**Example:** for *"uncrowded reef break on the east coast tomorrow morning"*, the baseline ranked a spot in the **south** first, while the improved system only returned **east-coast reef breaks**.

## 🚀 How to use it

**1. Clone the repo and install**

Clone the repo, open the folder in VS Code, then run in the terminal:

```powershell
git clone https://github.com/kollenur-student/MIS590_IR_Project.git
py -m venv .venv
.venv\Scripts\activate
pip install -r requirements.txt
```

**2. Add your CWA API key**

Get a free key at [opendata.cwa.gov.tw](https://opendata.cwa.gov.tw), copy `.env.example` to `.env` and paste it in.

> ⚠️ **Never commit `.env`.**

**3. Build the corpus and index (once)**

Run `01_spot_corpus.ipynb` and `02_indexing.ipynb`.

**4. Start the demo**

Open `notebooks/06_demo.ipynb`, select the **`.venv`** kernel and click **Run All**. Open the printed link (http://127.0.0.1:7860) in your browser. **Baseline is on the left, Improved on the right.**

**5. Try some queries**

- *"glassy waves for a newbie near Yilan on Saturday"*
- *"point break with board rental near Kenting"*

> 📖 The `README.md` has the full instructions.

## ❗ Important to know

- **The evaluation numbers right now are fake.** Notebook 05 runs on a mock smoke test, so don't report anything from it.
- **We still need to write the test collection ourselves:** ~50 queries in `eval/queries.tsv`, plus relevance judgments (0–3) in `eval/qrels.txt`. Templates and guidelines are in `eval/`.